# 01 · Attention from scratch — the prerequisite

> **Paper:** Appendix A.1 "Preliminaries: Multi-head attention layers", Eq. (3)–(8)

DETR is *"DEtection TRansformer"*. If the transformer half is a black box, the rest of this tutorial will feel like memorising, not understanding. So before we touch DETR, we build attention **from scratch, by hand**, and check our version against PyTorch's numerically.

**By the end you'll be able to answer:**
- What are Q, K, V, and why three of them?
- Why divide by √d?
- What's the difference between *self*-attention and *cross*-attention? (DETR's decoder uses both)
- Why does a transformer need positional encodings at all?
- Why must DETR's 100 object queries be *different from each other*?

Those last two questions are the whole reason notebooks `03` and `04` look the way they do.

**Nothing here needs a GPU or a download.** Small tensors, printed in full, so you can follow every number.

> **New to PyTorch itself?** This notebook assumes you can read `x.permute(2, 0, 1)` without flinching. If you can't yet, spend twenty minutes in [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) first — §2 (shape surgery) and §6 (matmul and the shape of attention) are the direct prerequisites for what follows.

In [1]:
%matplotlib inline
import torch, math
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=100)
print("torch", torch.__version__)

torch 2.14.0


## 1. The one-sentence intuition

> Attention is a **soft dictionary lookup.**

An ordinary Python dict is a *hard* lookup: `d["cat"]` returns exactly one value, and a key either matches or it doesn't.

Attention is the soft version:
- Every item offers a **key** (what it is) and a **value** (what it carries).
- You arrive with a **query** (what you're looking for).
- Instead of picking one key, you measure similarity against *all* keys, turn those similarities into weights that sum to 1, and return the **weighted blend of all values**.

| dict | attention |
|---|---|
| `key` | **K** — what each position advertises |
| `value` | **V** — what each position actually hands over |
| `d[q]` | **Q** — what this position is asking for |
| exact match | dot-product similarity → softmax |
| returns 1 value | returns a weighted average of all values |

In DETR's decoder this is literal: an object query asks *"is there an object with these properties?"* and gets back a blend of image features, weighted by how well each image patch answers.

## 2. A worked example with 4 tokens

We'll use a **sequence of 4 tokens, each 8-dimensional**. Small enough to print, big enough to be real.

In [3]:
L, D = 4, 8          # L = sequence length (4 tokens), D = embedding dim
x = torch.randn(L, D)
print("x — our input sequence")
print("shape:", tuple(x.shape), " = (tokens, features)")
print(x)

x — our input sequence
shape: (4, 8)  = (tokens, features)
tensor([[-0.614,  0.032, -0.493,  0.248,  0.440,  0.112,  0.641,  0.441],
        [-0.102,  0.792, -0.290,  0.053,  0.523,  2.302, -1.469, -1.587],
        [-0.673,  0.873,  1.055,  0.178, -0.230, -0.392,  0.543, -0.395],
        [-0.446,  0.744,  1.521,  3.411, -1.531, -1.234,  1.820, -0.552]])


### Step 1 — project into Q, K, V

Q, K and V are all **linear projections of the same input** (for self-attention). Three different learned matrices, so the same token can *ask* one thing, *advertise* another, and *carry* a third.

This is the "one weight matrix applied independently to each row" idea from notebook `03` — `nn.Linear` over a sequence.

In [6]:
W_q = nn.Linear(D, D, bias=False)
W_k = nn.Linear(D, D, bias=False)
W_v = nn.Linear(D, D, bias=False)

q = W_q(x)      # shape: (4, 8)  "what am I looking for?"
k = W_k(x)      # shape: (4, 8)  "what do I advertise?"
v = W_v(x)      # shape: (4, 8)  "what do I hand over?"

print("q:", tuple(q.shape), " k:", tuple(k.shape), " v:", tuple(v.shape))
print("\nThree different matrices -> three different views of the SAME 4 tokens.")
print("q[0] and k[0] both come from token 0, but they are not equal:")
print("  q[0] =", q[0])
print("  k[0] =", k[0])

q: (4, 8)  k: (4, 8)  v: (4, 8)

Three different matrices -> three different views of the SAME 4 tokens.
q[0] and k[0] both come from token 0, but they are not equal:
  q[0] = tensor([-0.204, -0.230, -0.094, -0.309,  0.014,  0.267, -0.018, -0.263], grad_fn=<SelectBackward0>)
  k[0] = tensor([ 0.165,  0.390, -0.381, -0.174,  0.052,  0.054,  0.023,  0.103], grad_fn=<SelectBackward0>)


In [9]:
W_k(x)

tensor([[ 0.165,  0.390, -0.381, -0.174,  0.052,  0.054,  0.023,  0.103],
        [-0.668,  0.205,  0.074, -0.097,  0.034,  0.491, -0.896, -0.405],
        [-0.024,  0.110,  0.406,  0.432, -0.533,  0.110,  0.256, -0.059],
        [ 0.839,  0.683,  0.541, -0.749, -1.832, -1.246, -0.205,  0.221]], grad_fn=<MmBackward0>)

### Step 2 — similarity scores: every query against every key

`scores[i, j]` = how much **token i's query** matches **token j's key**. A dot product: large when the two vectors point the same way.

In [10]:
scores = q @ k.T          # shape: (4, 4)   (L, D) @ (D, L) -> (L, L)
print("scores = q @ k.T   shape:", tuple(scores.shape))
print(scores)
print()
print("row i = token i's query scored against ALL keys")
print("scores[0, 2] is token 0 asking about token 2:", scores[0, 2].item())
print("  verify by hand:", torch.dot(q[0], k[2]).item())

scores = q @ k.T   shape: (4, 4)
tensor([[-0.046,  0.366, -0.159, -0.562],
        [-0.156, -0.901, -0.033, -1.877],
        [-0.047, -0.190,  0.717,  2.121],
        [ 0.822, -0.003,  0.753,  6.954]], grad_fn=<MmBackward0>)

row i = token i's query scored against ALL keys
scores[0, 2] is token 0 asking about token 2: -0.15930308401584625
  verify by hand: -0.15930306911468506


### Step 3 — scale by √d

Divide by `√D`. Here's *why*, demonstrated rather than asserted: the dot product of two random D-dim vectors has variance that **grows with D**. Large scores make softmax saturate — one weight goes to ~1.0, the rest to ~0 — and a saturated softmax has almost **zero gradient**, so the layer stops learning.

In [14]:
(a*b).shape

torch.Size([2000, 1024])

In [11]:
print("std of raw dot products as D grows:")
for d in [8, 64, 256, 1024]:
    a, b = torch.randn(2000, d), torch.randn(2000, d)
    raw = (a * b).sum(-1)
    print(f"   D={d:5d}:  std(q·k) = {raw.std():7.2f}   after /sqrt(D) = {(raw/math.sqrt(d)).std():.2f}")

print("\nEffect on softmax (one row of scores, scaled up vs down):")
row = torch.tensor([2.0, 1.0, 0.5, 0.2])
for mult, label in [(1, "well-scaled"), (10, "unscaled (large D)")]:
    w = (row * mult).softmax(-1)
    print(f"   {label:<20} softmax = {w}   max={w.max():.3f}")
print("\n-> unscaled: one weight ~1.0, gradient vanishes. That is what /sqrt(D) prevents.")

std of raw dot products as D grows:
   D=    8:  std(q·k) =    2.83   after /sqrt(D) = 1.00
   D=   64:  std(q·k) =    8.10   after /sqrt(D) = 1.01
   D=  256:  std(q·k) =   16.13   after /sqrt(D) = 1.01
   D= 1024:  std(q·k) =   33.30   after /sqrt(D) = 1.04

Effect on softmax (one row of scores, scaled up vs down):
   well-scaled          softmax = tensor([0.569, 0.209, 0.127, 0.094])   max=0.569
   unscaled (large D)   softmax = tensor([1.000, 0.000, 0.000, 0.000])   max=1.000

-> unscaled: one weight ~1.0, gradient vanishes. That is what /sqrt(D) prevents.


In [17]:
scaled = scores / math.sqrt(D)
print("scaled scores:")
print(scaled)

scaled scores:
tensor([[-0.016,  0.130, -0.056, -0.199],
        [-0.055, -0.319, -0.012, -0.663],
        [-0.017, -0.067,  0.254,  0.750],
        [ 0.291, -0.001,  0.266,  2.458]], grad_fn=<DivBackward0>)


### Step 4 — softmax into attention weights

Row-wise softmax. Each row becomes a **probability distribution over the 4 tokens**: "how much of my output should come from each one?"

In [18]:
attn = scaled.softmax(dim=-1)      # shape: (4, 4)
print("attention weights:")
print(attn)
print()
print("every ROW sums to 1 (it is a distribution over tokens):")
print("  row sums:", attn.sum(-1))
print("  column sums are NOT 1:", attn.sum(0), "<- a common confusion")

attention weights:
tensor([[0.253, 0.293, 0.243, 0.211],
        [0.298, 0.229, 0.311, 0.162],
        [0.185, 0.176, 0.242, 0.398],
        [0.087, 0.065, 0.085, 0.762]], grad_fn=<SoftmaxBackward0>)

every ROW sums to 1 (it is a distribution over tokens):
  row sums: tensor([1.000, 1.000, 1.000, 1.000], grad_fn=<SumBackward1>)
  column sums are NOT 1: tensor([0.823, 0.762, 0.881, 1.533], grad_fn=<SumBackward1>) <- a common confusion


### Step 5 — weighted sum of values

Finally, blend the values using those weights.

In [19]:
out = attn @ v            # shape: (4, 4) @ (4, 8) -> (4, 8)
print("output:", tuple(out.shape), "-- same shape as the input x")
print(out)
print()
print("token 0's output is a weighted blend of ALL 4 value vectors:")
manual = sum(attn[0, j] * v[j] for j in range(L))
print("  by hand :", manual)
print("  matmul  :", out[0])
print("  equal?  ", torch.allclose(manual, out[0], atol=1e-6))

output: (4, 8) -- same shape as the input x
tensor([[-0.118, -0.261, -0.148,  0.024, -0.103, -0.136,  0.158, -0.115],
        [-0.115, -0.239, -0.129,  0.052, -0.042, -0.078,  0.174, -0.126],
        [-0.399, -0.586, -0.168, -0.160, -0.148, -0.182,  0.335, -0.318],
        [-0.843, -1.111, -0.226, -0.505, -0.273, -0.369,  0.581, -0.608]], grad_fn=<MmBackward0>)

token 0's output is a weighted blend of ALL 4 value vectors:
  by hand : tensor([-0.118, -0.261, -0.148,  0.024, -0.103, -0.136,  0.158, -0.115], grad_fn=<AddBackward0>)
  matmul  : tensor([-0.118, -0.261, -0.148,  0.024, -0.103, -0.136,  0.158, -0.115], grad_fn=<SelectBackward0>)
  equal?   True


### The whole thing in one line

That's the entire attention mechanism — paper Eq. (8):

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

In [9]:
def attention(q, k, v):
    """Scaled dot-product attention. Returns (output, attention_weights)."""
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    w = scores.softmax(-1)
    return w @ v, w

mine, w = attention(q, k, v)
theirs = F.scaled_dot_product_attention(q, k, v)     # PyTorch's built-in

print("my implementation matches PyTorch:", torch.allclose(mine, theirs, atol=1e-6))
print("max abs difference:", (mine - theirs).abs().max().item())

my implementation matches PyTorch: True
max abs difference: 5.960464477539063e-08


## 3. Self-attention vs cross-attention

The single most useful distinction for reading DETR. The formula is **identical**; only *where Q, K, V come from* changes.

| | Q from | K, V from | In DETR |
|---|---|---|---|
| **Self**-attention | the sequence itself | the same sequence | encoder: image patches ↔ image patches<br>decoder: query ↔ query (kills duplicates) |
| **Cross**-attention | sequence A | sequence **B** | decoder: object queries → image features |

Cross-attention is how DETR's 100 object queries actually *see* the image.

In [10]:
# SELF-attention: 4 image tokens attending to each other
img = torch.randn(4, D)
o_self, w_self = attention(W_q(img), W_k(img), W_v(img))
print("self : Q,K,V all from img(4)  -> attn", tuple(w_self.shape), " out", tuple(o_self.shape))

# CROSS-attention: 2 object queries reading from 4 image tokens
queries = torch.randn(2, D)
o_cross, w_cross = attention(W_q(queries), W_k(img), W_v(img))
print("cross: Q from queries(2), K,V from img(4)")
print("       -> attn", tuple(w_cross.shape), " out", tuple(o_cross.shape))
print()
print("KEY: output length follows the QUERY, not the keys.")
print("  2 queries in -> 2 outputs out, no matter how many image tokens there are.")
print("  That is exactly how DETR turns 850 image tokens into 100 detections.")

self : Q,K,V all from img(4)  -> attn (4, 4)  out (4, 8)
cross: Q from queries(2), K,V from img(4)
       -> attn (2, 4)  out (2, 8)

KEY: output length follows the QUERY, not the keys.
  2 queries in -> 2 outputs out, no matter how many image tokens there are.
  That is exactly how DETR turns 850 image tokens into 100 detections.


Re-read that last point — it is the structural reason DETR works. The decoder's output has one row **per query**, so N queries always produce exactly N predictions, regardless of image size. That's the "fixed set of N predictions" from the paper's §3.1.

## 4. Multi-head attention

One attention operation computes **one** kind of relationship. Real transformers run several in parallel — "heads" — each with its own Q/K/V projections, and concatenate the results.

The trick: instead of `nheads` separate D-dimensional attentions (expensive), **split** the D dimensions into `nheads` chunks of `D / nheads`. Same total compute, multiple independent relationships.

DETR uses `nheads=8` with `d_model=256`, so each head is 32-dimensional.

In [11]:
nheads = 2
head_dim = D // nheads
print(f"D={D}, nheads={nheads} -> head_dim={head_dim}")

def split_heads(t, nheads):
    L, D_ = t.shape
    return t.view(L, nheads, D_ // nheads).transpose(0, 1)   # (nheads, L, head_dim)

qh, kh, vh = split_heads(q, nheads), split_heads(k, nheads), split_heads(v, nheads)
print("after split:", tuple(qh.shape), " = (heads, tokens, head_dim)")

out_h, w_h = attention(qh, kh, vh)       # attention works on the batched dims for free
print("per-head output:", tuple(out_h.shape))

merged = out_h.transpose(0, 1).reshape(L, D)   # concat heads back
print("after concat   :", tuple(merged.shape), "-- back to the original shape")
print()
print("the two heads learned DIFFERENT attention patterns:")
print("head 0, token 0:", w_h[0, 0])
print("head 1, token 0:", w_h[1, 0])

D=8, nheads=2 -> head_dim=4
after split: (2, 4, 4)  = (heads, tokens, head_dim)
per-head output: (2, 4, 4)
after concat   : (4, 8) -- back to the original shape

the two heads learned DIFFERENT attention patterns:
head 0, token 0: tensor([0.258, 0.267, 0.192, 0.282], grad_fn=<SelectBackward0>)
head 1, token 0: tensor([0.359, 0.169, 0.333, 0.139], grad_fn=<SelectBackward0>)


### Verify against `nn.MultiheadAttention`

The real test: reimplement PyTorch's module by hand, pulling out its actual weights, and check the numbers match. `nn.MultiheadAttention` packs Wq/Wk/Wv into one `in_proj_weight` of shape `(3D, D)`.

In [12]:
mha = nn.MultiheadAttention(embed_dim=D, num_heads=nheads, bias=True)
mha.eval()

seq = torch.randn(L, 1, D)          # (L, N, D) -- DETR's sequence-first convention
ref_out, ref_w = mha(seq, seq, seq)   # self-attention

# --- now do it by hand with mha's own weights ---
Wi, bi = mha.in_proj_weight, mha.in_proj_bias      # (3D, D), (3D,)
Wq_, Wk_, Wv_ = Wi.chunk(3, dim=0)
bq_, bk_, bv_ = bi.chunk(3, dim=0)

s = seq[:, 0, :]                                   # drop batch -> (L, D)
qm, km, vm = s @ Wq_.T + bq_, s @ Wk_.T + bk_, s @ Wv_.T + bv_
om, wm = attention(split_heads(qm, nheads), split_heads(km, nheads), split_heads(vm, nheads))
om = om.transpose(0, 1).reshape(L, D)
om = om @ mha.out_proj.weight.T + mha.out_proj.bias   # final output projection

print("PyTorch output :", ref_out[:, 0, :][0])
print("my output      :", om[0])
print()
print("outputs match           :", torch.allclose(ref_out[:, 0, :], om, atol=1e-5))
print("attention weights match :", torch.allclose(ref_w[0], wm.mean(0), atol=1e-5))
print("   (nn.MultiheadAttention AVERAGES the heads' weights before returning them --")
print("    that is why notebook 05's attention maps are head-averaged.)")

PyTorch output : tensor([-0.348,  0.050, -0.081,  0.130,  0.204,  0.109,  0.107, -0.506], grad_fn=<SelectBackward0>)
my output      : tensor([-0.348,  0.050, -0.081,  0.130,  0.204,  0.109,  0.107, -0.506], grad_fn=<SelectBackward0>)

outputs match           : True
attention weights match : True
   (nn.MultiheadAttention AVERAGES the heads' weights before returning them --
    that is why notebook 05's attention maps are head-averaged.)


## 5. The punchline: attention is permutation-invariant

This is the property that shapes DETR's entire design. **Shuffle the input tokens and the outputs are just as shuffled — no information about order or position survives.**

Let's prove it.

In [13]:
perm = torch.tensor([2, 0, 3, 1])         # some shuffle

o_orig, _ = attention(W_q(x), W_k(x), W_v(x))          # attention on the original order
xs = x[perm]                                            # shuffle the tokens
o_shuf, _ = attention(W_q(xs), W_k(xs), W_v(xs))       # attention on the shuffled order

print("output of shuffled input == shuffled output of original input?")
print("  ", torch.allclose(o_shuf, o_orig[perm], atol=1e-6))
print()
print("So attention has NO idea which token came first.")
print("To it, a sequence is an unordered BAG of tokens.")

output of shuffled input == shuffled output of original input?
   True

So attention has NO idea which token came first.
To it, a sequence is an unordered BAG of tokens.


### Two consequences, and they are the whole of notebooks 02 and 03

**(a) You must inject position manually.** An image flattened into 850 tokens loses all 2-D layout. Detection is *entirely* about where things are — so DETR adds a **positional encoding** to every token. That is notebook `03` §5.

**(b) The N object queries must differ from one another.** If all 100 queries were identical vectors, permutation-invariance means they'd all produce **identical outputs** — 100 copies of the same box. The queries are *learned to be different*, which is what lets them specialise. That is notebook `04` §2.

Let's verify (b) directly, since it's the one people find surprising:

In [14]:
img_tokens = torch.randn(6, D)

same = torch.randn(1, D).repeat(3, 1)     # 3 IDENTICAL queries
o_same, _ = attention(W_q(same), W_k(img_tokens), W_v(img_tokens))
print("3 identical queries -> are the 3 outputs identical?")
print("   out[0] == out[1]:", torch.allclose(o_same[0], o_same[1], atol=1e-6))
print("   -> 3 duplicate detections. Useless.\n")

diff = torch.randn(3, D)                  # 3 DIFFERENT queries
o_diff, _ = attention(W_q(diff), W_k(img_tokens), W_v(img_tokens))
print("3 different queries -> outputs differ?")
print("   out[0] == out[1]:", torch.allclose(o_diff[0], o_diff[1], atol=1e-6))
print("   -> 3 distinct detections. This is why query_embed is LEARNED.")

3 identical queries -> are the 3 outputs identical?
   out[0] == out[1]: True
   -> 3 duplicate detections. Useless.

3 different queries -> outputs differ?
   out[0] == out[1]: False
   -> 3 distinct detections. This is why query_embed is LEARNED.


> *"Since the decoder is also permutation-invariant, the N input embeddings must be different to produce different results."* — paper §3.2

You have now derived that sentence yourself, rather than taking it on faith.

## 6. Exercises

Try each before opening the solution.

---

**Exercise 1.** Given `attn` of shape `(4, 4)` from §2, what does `attn[2].argmax()` tell you? What about `attn[:, 2].argmax()`?

<details><summary>Solution</summary>

`attn[2].argmax()` — which token **token 2 attends to most** (row = one query's distribution, sums to 1).

`attn[:, 2].argmax()` — which token attends most **to token 2**. This is *not* a distribution and does not sum to 1. Rows and columns mean different things; mixing them up is the single most common attention bug.

```python
print("token 2 looks mostly at token", attn[2].argmax().item())
print("token 2 is looked at most by token", attn[:, 2].argmax().item())
```
</details>

---

**Exercise 2.** In cross-attention with 5 queries and 900 image tokens, what shape is the attention weight matrix, and what shape is the output?

<details><summary>Solution</summary>

Weights `(5, 900)`, output `(5, D)`.

Output length always follows the **query**. This is exactly notebook `06`, where decoder cross-attention was `(100, 850)`: 100 queries × 850 image tokens.
</details>

---

**Exercise 3.** Remove the `/ √d` scaling from `attention()` and rerun §2 with `D = 512` instead of 8. What happens to the attention weights, and why does that stop the model learning?

<details><summary>Solution</summary>

The weights collapse to nearly one-hot (max ≈ 1.0). Softmax's gradient is `p(1-p)`, so when `p → 1` the gradient → 0 and no signal flows back. Scaling keeps scores in a range where softmax stays soft and differentiable.

```python
x2 = torch.randn(4, 512)
Wq2 = nn.Linear(512, 512, bias=False)
s = Wq2(x2) @ Wq2(x2).T            # unscaled
print("unscaled max weight:", s.softmax(-1).max().item())
print("  scaled max weight:", (s / math.sqrt(512)).softmax(-1).max().item())
```
</details>

---

**Exercise 4.** DETR's decoder adds the object query to Q and K, but **not** to V (notebook `03` §5). Using the dictionary analogy, why is that the right choice?

<details><summary>Solution</summary>

Q and K decide **where to look** — position should influence that. V is **what gets handed back** — that should be image content, not position. Mixing position into V would contaminate the retrieved features with coordinates.

The paper ablates this in Table 3: passing encodings into the attention layers beats adding them once at the input by 1.4 AP.
</details>

---

**Exercise 5 (harder).** Write `attention()` so it accepts a `key_padding_mask` of shape `(L,)` where `True` = padding to ignore. Verify masked positions get exactly zero weight.

<details><summary>Solution</summary>

Set masked scores to `-inf` **before** the softmax, so they become exactly 0 after it.

```python
def attention_masked(q, k, v, key_padding_mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if key_padding_mask is not None:
        scores = scores.masked_fill(key_padding_mask, float("-inf"))
    w = scores.softmax(-1)
    return w @ v, w

mask = torch.tensor([False, False, True, True])    # ignore last 2 tokens
_, w = attention_masked(q, k, v, mask)
print(w)
print("masked columns are exactly zero:", bool((w[:, 2:] == 0).all()))
```

This is precisely what DETR does with the padding masks from notebook `03` §1 — real images padded into a batch must not attend to the padding.
</details>

## You're ready

| You now know | Where DETR uses it |
|---|---|
| Q, K, V and scaled dot-product attention | everywhere |
| output length follows the **query** | 100 queries → 100 detections |
| self- vs cross-attention | encoder vs decoder |
| multi-head, and that returned weights are head-averaged | notebook `06`'s attention maps |
| attention is **permutation-invariant** | why positional encodings (`03`) and distinct queries (`04`) exist |
| padding masks via `-inf` before softmax | batching variable-sized images (`03`) |

Continue to **`02`** — why object detection needed a transformer in the first place.